# Наочний приклад: Pydantic — валідація даних для Python та AI

Ви вже знаєте ООП: клас, атрибути, `__init__`, типи. **Pydantic** — це бібліотека, яка бере звичайний клас і додає до нього **автоматичну перевірку (валідацію) даних** за анотаціями типів.

Навіщо це в AI engineering:

- 🤖 **Структуровані виходи LLM** — модель повертає JSON, а Pydantic гарантує, що він має правильну форму.
- 🛠️ **Аргументи інструментів (tools)** для агентів — описуємо, які параметри приймає інструмент.
- 🌐 **API** (FastAPI цілком побудований на Pydantic) — валідація запитів і відповідей.
- ⚙️ **Конфігурації** — безпечне читання налаштувань і ключів.

Ми пройдемо крок за кроком:

1. Звичайний клас проти Pydantic-моделі
2. Типи, автоматичне приведення та помилки валідації
3. `Field` — значення за замовчуванням і обмеження
4. Вкладені моделі
5. Власні валідатори
6. Серіалізація: модель ↔ dict ↔ JSON
7. 🤖 Практика: структурований вихід LLM

> 💡 Запускайте клітинки по черзі. Експериментуйте — підставляйте «погані» дані й дивіться на помилки!

---
## 0. Встановлення

Якщо Pydantic ще не встановлено — розкоментуйте рядок нижче. У цьому проєкті вже стоїть **Pydantic v2** (увесь урок на синтаксисі v2).

In [ ]:
# %pip install pydantic

import pydantic
print("Pydantic версія:", pydantic.VERSION)

---
## 1. Звичайний клас проти Pydantic-моделі 🆚

Згадаймо, як ми писали клас вручну: треба самим прописати `__init__`, присвоїти кожен атрибут, а **жодної перевірки типів немає** — Python радо прийме будь-що.

In [ ]:
class UserPlain:
    def __init__(self, name, age):
        self.name = name
        self.age = age


# Жодних перевірок: вік рядком? Будь ласка. Помилка спливе пізніше й у дивному місці.
u = UserPlain("Іван", "двадцять")
print(u.name, u.age, type(u.age))

Тепер те саме на Pydantic. Успадковуємо `BaseModel` і **лише описуємо поля з типами** — `__init__` Pydantic згенерує сам (як `@dataclass`, але з валідацією).

In [ ]:
from pydantic import BaseModel


class User(BaseModel):
    name: str
    age: int


u = User(name="Іван", age=20)
print(u)            # гарний __repr__ безкоштовно
print(u.name, u.age, type(u.age))

---
## 2. Валідація: приведення типів і помилки ⚠️

Pydantic **намагається розумно привести** значення до потрібного типу. Рядок `"20"` стане числом `20`. Але якщо привести **неможливо** — отримаємо зрозумілу помилку `ValidationError`, ще на етапі створення об'єкта.

In [ ]:
# "20" (рядок) -> 20 (int): коректне приведення
u = User(name="Іван", age="20")
print(u.age, type(u.age))

In [ ]:
from pydantic import ValidationError

# "двадцять" привести до int НЕ можна -> зрозуміла помилка
try:
    User(name="Іван", age="двадцять")
except ValidationError as e:
    print(e)

> 🔑 Головна ідея: **погані дані не потраплять усередину програми**. Помилка ловиться одразу, на «кордоні», а не десь глибоко в логіці через 100 рядків.

---
## 3. `Field`: значення за замовчуванням та обмеження 🎚️

За допомогою `Field(...)` можна задати:

- значення за замовчуванням (поле стає необов'язковим);
- обмеження: `ge`/`le` (≥/≤), `gt`/`lt` (>/<), `min_length`/`max_length`;
- `description` — опис поля (LLM його "бачить", коли модель формує структурований вихід!).

`Optional[...]` (або `... | None`) означає, що поле може бути `None`.

In [ ]:
from typing import Optional
from pydantic import BaseModel, Field


class Product(BaseModel):
    name: str = Field(min_length=1, description="Назва товару")
    price: float = Field(gt=0, description="Ціна, має бути більшою за 0")
    quantity: int = Field(default=1, ge=0, description="Кількість на складі")
    discount: Optional[float] = Field(default=None, ge=0, le=100)


p = Product(name="Клавіатура", price=1200)
print(p)

In [ ]:
# Порушуємо обмеження: ціна <= 0 і знижка > 100
try:
    Product(name="Миша", price=-50, discount=150)
except ValidationError as e:
    print(e)

Зверніть увагу: Pydantic зібрав **усі** помилки одразу (і ціну, і знижку), а не впав на першій.

---
## 4. Вкладені моделі 🧩

Поле моделі саме може бути моделлю — це як **композиція (has-a)** з ООП. Pydantic перевірить структуру на всю глибину. Списки моделей теж працюють: `list[Item]`.

In [ ]:
class Address(BaseModel):
    city: str
    street: str


class OrderItem(BaseModel):
    product: str
    qty: int = Field(gt=0)


class Order(BaseModel):
    customer: str
    address: Address                 # вкладена модель (has-a)
    items: list[OrderItem]           # список моделей


# Pydantic сам перетворить вкладені dict у відповідні моделі
order = Order(
    customer="Олена",
    address={"city": "Київ", "street": "Хрещатик 1"},
    items=[
        {"product": "Зошит", "qty": 3},
        {"product": "Ручка", "qty": 10},
    ],
)
print(order.address.city)
print(order.items[0].product, order.items[0].qty)
print(type(order.address), type(order.items[0]))

---
## 5. Власні валідатори ✅

Коли вбудованих обмежень `Field` не вистачає, пишемо **власну перевірку** методом із декоратором `@field_validator`. Він спрацьовує при створенні об'єкта; якщо щось не так — піднімаємо `ValueError`, і Pydantic загорне її у `ValidationError`.

In [ ]:
from pydantic import field_validator


class Account(BaseModel):
    username: str
    email: str

    @field_validator("username")
    @classmethod
    def username_no_spaces(cls, v: str) -> str:
        if " " in v:
            raise ValueError("username не може містити пробілів")
        return v.lower()              # валідатор може ще й нормалізувати значення

    @field_validator("email")
    @classmethod
    def email_must_have_at(cls, v: str) -> str:
        if "@" not in v:
            raise ValueError("email має містити '@'")
        return v


print(Account(username="IvanUA", email="ivan@mail.com"))   # username стане 'ivanua'

try:
    Account(username="ivan ua", email="bad-email")
except ValidationError as e:
    print(e)

---
## 6. Серіалізація: модель ↔ dict ↔ JSON 🔄

Це найважливіше для роботи з API та LLM:

- `model_dump()` → перетворює модель на `dict`;
- `model_dump_json()` → одразу на JSON-рядок;
- `Model(**dict)` або `Model.model_validate(dict)` → з даних назад у модель (з валідацією!);
- `Model.model_validate_json(json_str)` → з JSON-рядка у модель.

In [ ]:
p = Product(name="Монітор", price=8000, quantity=5)

as_dict = p.model_dump()
print("dict:", as_dict)

as_json = p.model_dump_json(indent=2)
print("JSON:\n", as_json)

# Назад із JSON-рядка у модель (типова ситуація: відповідь від API / LLM)
raw = '{"name": "Веб-камера", "price": "950", "quantity": 2}'
p2 = Product.model_validate_json(raw)
print("Відновлено:", p2, "| price тип:", type(p2.price))

---
## 7. 🤖 Практика: структурований вихід LLM

Найпоширеніший сценарій в AI engineering. Ми просимо LLM **повернути JSON**, а Pydantic-модель виступає **контрактом**: вона і описує бажану форму (через типи й `description`), і валідує те, що реально повернула модель.

Уявімо інструмент, що витягує структуру з вільного тексту відгуку про фільм.

In [ ]:
from typing import Literal


class MovieReview(BaseModel):
    """Структура, яку ми хочемо отримати від LLM."""

    title: str = Field(description="Назва фільму")
    rating: int = Field(ge=1, le=10, description="Оцінка від 1 до 10")
    sentiment: Literal["позитивний", "негативний", "нейтральний"]
    keywords: list[str] = Field(default_factory=list, description="Ключові слова")


# Так виглядає JSON-схема, яку зручно віддати моделі в інструкції / tool-описі:
import json
print(json.dumps(MovieReview.model_json_schema(), ensure_ascii=False, indent=2))

In [ ]:
# Уявімо, що це сирий JSON-рядок, який повернула LLM:
llm_output = '''
{
  "title": "Інтерстеллар",
  "rating": 9,
  "sentiment": "позитивний",
  "keywords": ["космос", "час", "емоції"]
}
'''

review = MovieReview.model_validate_json(llm_output)
print(review)
print("Оцінка:", review.rating, "| Настрій:", review.sentiment)

In [ ]:
# А якщо LLM "зґалюцинувала" і повернула rating=15 (поза межами 1..10)?
bad_output = '{"title": "X", "rating": 15, "sentiment": "дивний"}'

try:
    MovieReview.model_validate_json(bad_output)
except ValidationError as e:
    print("Спіймали некоректний вихід LLM:\n")
    print(e)

> 🔑 Саме так Pydantic захищає AI-застосунок: навіть якщо модель помилилася, **некоректні дані не пройдуть далі** — ми це помітимо й зможемо повторити запит або обробити помилку.

---
## Підсумок 📌

| Інструмент | Що робить |
|---|---|
| `BaseModel` | базовий клас моделі; `__init__` і валідація — автоматично |
| анотації типів | задають, який тип має поле; Pydantic приводить і перевіряє |
| `Field(...)` | значення за замовчуванням, обмеження (`ge`, `le`, `min_length`...), `description` |
| `Optional[T]` / `T \| None` | поле може бути `None` |
| вкладені моделі | композиція (has-a); валідація на всю глибину |
| `@field_validator` | власна перевірка/нормалізація поля |
| `model_dump()` / `model_dump_json()` | модель → dict / JSON |
| `model_validate()` / `model_validate_json()` | dict / JSON → модель (з валідацією) |
| `model_json_schema()` | JSON-схема — контракт для LLM та tools |

**Зв'язок з ООП:** Pydantic-модель — це звичайний клас (успадкування від `BaseModel`), де анотації типів + `Field` задають «контракт» (абстракція), а вкладені моделі — це композиція.

### 🎯 Завдання для самоперевірки

1. Створіть модель `Student` з полями `name: str`, `group: str`, `grades: list[int]` (кожна оцінка 1..12). Додайте валідатор, що забороняє порожній список оцінок.
2. Додайте властивість/метод `average`, що повертає середній бал.
3. Створіть модель `WeatherReport` (місто, температура, опис) і "розпарсіть" нею ось такий JSON: `'{"city": "Львів", "temperature": "15", "description": "хмарно"}'`.

➡️ Далі: спробуйте під'єднати Pydantic до реального виклику LLM (структуровані виходи) у вашому проєкті — наприклад, у `services/llm_factory.py`.